# Semana 4: Normalización e Integridad Referencial
## Notebook de Ejercicios (NB2) – Práctica en SQL Fiddle

### Objetivo
Llevar a la práctica los conceptos de normalización (hasta 3FN) e integridad referencial implementando un modelo relacional en **SQL Fiddle**. Partiremos de un enunciado, diseñaremos las tablas normalizadas, definiremos claves primarias y foráneas, y probaremos la integridad insertando datos que violen las restricciones.

### Herramienta
Utilizaremos [DB-Fiddle](https://www.db-fiddle.com/) (SQL Fiddle) con motor **PostgreSQL**.

### Enunciado

Una empresa de ventas online necesita una base de datos para gestionar sus operaciones. Los requisitos son:

*   **Clientes:** Cada cliente tiene un ID único, nombre, email (único), teléfono y dirección.
*   **Productos:** Cada producto tiene un código de producto único, nombre, descripción, precio actual y stock.
*   **Pedidos:** Un pedido lo realiza un cliente. Tiene un número de pedido único, fecha del pedido, estado (pendiente, enviado, entregado).
*   **Detalle de Pedido:** Un pedido puede contener varios productos, con la cantidad y el precio en el momento de la compra (precio_unitario).
*   **Categorías:** Los productos pertenecen a categorías. Una categoría tiene un ID y un nombre. Un producto pertenece a una sola categoría.

**Nota:** Asumimos que el precio de un producto puede cambiar con el tiempo, por lo que en DetallePedido se guarda el precio del momento (precio_unitario).

## Paso 1: Diseño del Modelo Lógico Normalizado (3FN)

Aplicando las formas normales (1FN, 2FN, 3FN), obtenemos las siguientes tablas:

```
Cliente (id_cliente, nombre, email, telefono, direccion)
Categoria (id_categoria, nombre)
Producto (id_producto, nombre, descripcion, precio, stock, id_categoria)
Pedido (id_pedido, fecha, estado, id_cliente)
DetallePedido (id_pedido, id_producto, cantidad, precio_unitario)
```

**Claves y restricciones:**
*   `Cliente.id_cliente` PK
*   `Cliente.email` UNIQUE
*   `Categoria.id_categoria` PK
*   `Producto.id_producto` PK, `Producto.id_categoria` FK a `Categoria.id_categoria`
*   `Pedido.id_pedido` PK, `Pedido.id_cliente` FK a `Cliente.id_cliente`
*   `DetallePedido` PK compuesta `(id_pedido, id_producto)`, FK a `Pedido.id_pedido` y a `Producto.id_producto`
*   `Pedido.estado` debe ser uno de ('pendiente', 'enviado', 'entregado') (CHECK)
*   `DetallePedido.cantidad` > 0 (CHECK)
*   `Producto.precio` > 0 (CHECK)
*   `Producto.stock` >= 0 (CHECK)

## Paso 2: Implementación en SQL Fiddle

Abre [DB-Fiddle](https://www.db-fiddle.com/), selecciona **PostgreSQL** y ejecuta los siguientes scripts.

### 2.1. Creación de tablas (Schema)

Copia y pega el siguiente código en el panel superior (Schema) y ejecuta.

In [ ]:
# Código SQL para crear las tablas (PostgreSQL)
sql_create = """
-- Tabla Categoria
CREATE TABLE categoria (
    id_categoria SERIAL PRIMARY KEY,
    nombre VARCHAR(100) NOT NULL
);

-- Tabla Cliente
CREATE TABLE cliente (
    id_cliente SERIAL PRIMARY KEY,
    nombre VARCHAR(100) NOT NULL,
    email VARCHAR(100) UNIQUE NOT NULL,
    telefono VARCHAR(20),
    direccion VARCHAR(200)
);

-- Tabla Producto
CREATE TABLE producto (
    id_producto SERIAL PRIMARY KEY,
    nombre VARCHAR(100) NOT NULL,
    descripcion TEXT,
    precio DECIMAL(10,2) NOT NULL CHECK (precio > 0),
    stock INT NOT NULL CHECK (stock >= 0),
    id_categoria INT NOT NULL REFERENCES categoria(id_categoria)
);

-- Tabla Pedido
CREATE TABLE pedido (
    id_pedido SERIAL PRIMARY KEY,
    fecha DATE NOT NULL DEFAULT CURRENT_DATE,
    estado VARCHAR(20) NOT NULL CHECK (estado IN ('pendiente', 'enviado', 'entregado')),
    id_cliente INT NOT NULL REFERENCES cliente(id_cliente)
);

-- Tabla DetallePedido
CREATE TABLE detalle_pedido (
    id_pedido INT REFERENCES pedido(id_pedido),
    id_producto INT REFERENCES producto(id_producto),
    cantidad INT NOT NULL CHECK (cantidad > 0),
    precio_unitario DECIMAL(10,2) NOT NULL,
    PRIMARY KEY (id_pedido, id_producto)
);
"""
print("✅ Código SQL para crear tablas listo. Cópialo y pruébalo en DB-Fiddle.")

### 2.2. Inserción de datos válidos

Pega el siguiente código en el panel inferior (Query) para insertar datos de prueba que cumplan las restricciones.

In [ ]:
# Código SQL para insertar datos válidos
sql_insert_valid = """
-- Insertar categorías
INSERT INTO categoria (nombre) VALUES ('Electrónica'), ('Ropa'), ('Hogar');

-- Insertar clientes
INSERT INTO cliente (nombre, email, telefono, direccion) VALUES
    ('Carlos López', 'carlos@email.com', '555-1111', 'Calle 1'),
    ('Ana Martínez', 'ana@email.com', '555-2222', 'Calle 2');

-- Insertar productos
INSERT INTO producto (nombre, descripcion, precio, stock, id_categoria) VALUES
    ('Laptop', 'Laptop de alta gama', 1200.00, 10, 1),
    ('Mouse', 'Mouse inalámbrico', 25.50, 50, 1),
    ('Camiseta', 'Camiseta de algodón', 15.00, 100, 2);

-- Insertar pedido
INSERT INTO pedido (fecha, estado, id_cliente) VALUES
    ('2025-03-10', 'pendiente', 1);

-- Insertar detalle del pedido
INSERT INTO detalle_pedido (id_pedido, id_producto, cantidad, precio_unitario) VALUES
    (1, 1, 1, 1200.00),
    (1, 2, 2, 25.50);

-- Verificar datos
SELECT * FROM cliente;
SELECT * FROM pedido;
SELECT * FROM detalle_pedido;
"""
print("✅ Código de inserciones válidas listo.")

### 2.3. Pruebas de integridad (violaciones)

Intenta ejecutar las siguientes sentencias que **deberían fallar** debido a las constraints. Esto demuestra que la integridad está funcionando.

**Inserta en DB-Fiddle y observa los errores.**

In [ ]:
# Código SQL para probar violaciones
sql_violations = """
-- 1. Violación de FK: insertar pedido con cliente que no existe
INSERT INTO pedido (fecha, estado, id_cliente) VALUES ('2025-03-11', 'pendiente', 999);

-- 2. Violación de FK: detalle de pedido con producto que no existe
INSERT INTO detalle_pedido (id_pedido, id_producto, cantidad, precio_unitario) VALUES (1, 999, 1, 10.00);

-- 3. Violación de CHECK: cantidad negativa en detalle
INSERT INTO detalle_pedido (id_pedido, id_producto, cantidad, precio_unitario) VALUES (1, 1, -5, 10.00);

-- 4. Violación de CHECK: precio de producto negativo
INSERT INTO producto (nombre, precio, stock, id_categoria) VALUES ('Producto malo', -10, 5, 1);

-- 5. Violación de UNIQUE: email duplicado en cliente
INSERT INTO cliente (nombre, email) VALUES ('Otro Carlos', 'carlos@email.com');

-- 6. Violación de PK: duplicar detalle de pedido
INSERT INTO detalle_pedido (id_pedido, id_producto, cantidad, precio_unitario) VALUES (1, 1, 3, 1200.00);
"""
print("✅ Código de violaciones listo. Pégarlo en DB-Fiddle y verifica los errores.")

## Paso 3: Actividad para el Estudiante - Nuevo Enunciado

Ahora, aplica el mismo proceso al siguiente enunciado. Deberás diseñar el modelo lógico normalizado hasta 3FN, crear las tablas en SQL Fiddle y probar la integridad.

### Enunciado: Sistema de Gestión de Biblioteca (ampliado)

Una biblioteca necesita gestionar sus libros, autores, socios y préstamos. Los requisitos son:

*   **Libros:** ISBN (único), título, año de publicación, editorial. Un libro puede tener varios autores.
*   **Autores:** ID único, nombre, nacionalidad, fecha de nacimiento.
*   **Socios:** Número de socio único, nombre, dirección, teléfono, email (único).
*   **Préstamos:** Un préstamo lo realiza un socio en una fecha determinada. Puede incluir varios libros (con fecha de devolución prevista y fecha de devolución real).
*   **Editoriales:** ID único, nombre, país.

**Notas:**
*   Un libro pertenece a una única editorial.
*   La relación entre libro y autor es N:M.
*   Un préstamo puede tener múltiples libros (detalle de préstamo).
*   Se debe registrar la fecha de préstamo, fecha prevista de devolución y fecha de devolución real (puede ser nula si aún no se ha devuelto).

### Tu tarea:

1.  **Diseña el modelo lógico** (tablas con columnas, PK, FK). Escríbelo en la celda de texto/markdown abajo.
2.  **Implementa las tablas en SQL Fiddle** (puedes usar PostgreSQL).
3.  **Define las constraints** (PK, FK, UNIQUE, CHECK).
4.  **Inserta datos de prueba** que sean válidos.
5.  **Prueba inserciones que violen** las restricciones y documenta los errores.

**Comparte tu código SQL en las siguientes celdas.**

In [ ]:
# Escribe aquí tu modelo lógico (como comentario)
# Ejemplo:
# Editorial (id_editorial, nombre, pais)
# Autor (id_autor, nombre, nacionalidad, fecha_nac)
# Libro (isbn, titulo, anio, id_editorial)
# Socio (id_socio, nombre, direccion, telefono, email)
# Prestamo (id_prestamo, fecha_prestamo, fecha_devolucion_prevista, fecha_devolucion_real, id_socio)
# Libro_Autor (id_autor, isbn)
# Detalle_Prestamo (id_prestamo, isbn)

print("Completa con tu diseño.")

In [ ]:
# Pega aquí tu código SQL de creación de tablas para la biblioteca
sql_biblioteca = """
-- Escribe aquí las sentencias CREATE TABLE
"""
print("Listo para copiar a DB-Fiddle.")

In [ ]:
# Pega aquí tus inserciones de prueba y violaciones
sql_biblioteca_pruebas = """
-- Datos válidos y violaciones
"""
print("Listo.")

---
## Ejercicios para el Estudiante

1.  **Añade más constraints:** En la biblioteca, añade un `CHECK` para que `fecha_devolucion_real` sea posterior a `fecha_prestamo` si no es nula. Pruébalo.

2.  **Crea una consulta compleja:** Escribe una consulta SQL que muestre los préstamos actuales (no devueltos) con el nombre del socio y los títulos de los libros.

3.  **Reflexiona sobre normalización:** ¿Está tu modelo de biblioteca en 3FN? Justifica por qué sí o por qué no, identificando posibles dependencias funcionales.

4.  **Compara con desnormalización:** ¿Cómo diseñarías una tabla desnormalizada para la biblioteca que permita consultas rápidas de préstamos? ¿Qué ventajas y desventajas tendría?

---
## Conclusiones de la Sesión

*   Hemos implementado un modelo relacional normalizado hasta **3FN** en SQL Fiddle, traduciendo un enunciado a tablas con claves primarias y foráneas.
*   Definimos **constraints** (`PRIMARY KEY`, `FOREIGN KEY`, `CHECK`, `UNIQUE`) para garantizar la integridad de los datos.
*   Insertamos datos válidos y probamos **violaciones de integridad**, observando los errores que el motor SQL devuelve.
*   Aplicamos el mismo proceso a un nuevo enunciado (biblioteca), reforzando el aprendizaje.
*   Discutimos la importancia de la normalización para evitar redundancias y mantener la consistencia.

¡Ahora sabes diseñar e implementar bases de datos normalizadas con integridad referencial!